In [0]:
dbutils.widgets.dropdown(name = 'Environment', defaultValue = 'dev', choices = ['dev','qa','prd'], label = 'Environment')


In [0]:
Env = dbutils.widgets.get("Environment")
print(Env)

In [0]:
silverTableName = f"saleslake_{Env}.silver_{Env}.cleaneddiscount"
print(silverTableName)

bronzeTableName = f"saleslake_{Env}.bronze_{Env}.rawdiscount"
print(bronzeTableName)

srcFileLoc=f"/Volumes/saleslake_{Env}/silver_{Env}/vol_saleslake_src_files_{Env}/daily_discount/"
print(srcFileLoc)

In [0]:
TableName= f"saleslake_{Env}.bronze_{Env}.rawdiscount"
print(TableName)
srcFileLoc = f"/Volumes/saleslake_{Env}/bronze_{Env}/vol_saleslake_src_files_{Env}/daily_discount"
print(srcFileLoc)


In [0]:
spark.sql(f"""
INSERT INTO {silverTableName}
SELECT DISTINCT
    UPPER(TRIM(discount_id)) AS discount_id,
    UPPER(TRIM(discount_code)) AS discount_code,
    UPPER(TRIM(discount_name)) AS discount_name,
    UPPER(TRIM(discount_type)) AS discount_type,
    CAST(TRIM(discount_value) AS DOUBLE) AS discount_value,
    CAST(TRIM(min_purchase_amount) AS DOUBLE) AS min_purchase_amount,
    CAST(TRIM(max_discount_amount) AS DOUBLE) AS max_discount_amount,
    TO_DATE(TRIM(valid_from), 'yyyy-MM-dd') AS valid_from,
    CURRENT_TIMESTAMP() AS ingest_ts
FROM {bronzeTableName}
WHERE ingest_ts >
(
    SELECT COALESCE(
        MAX(ingest_ts),
        TO_TIMESTAMP('1990-01-01', 'yyyy-MM-dd')
    )
    FROM {silverTableName}
)
ORDER BY discount_id
""")

In [0]:
%sql
--SELECT * FROM saleslake_dev.silver_dev.cleaneddiscount;